In [ ]:
!pip install -U transformers datasets accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 144.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 50.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=981f2617a4275150b6dbbf0592e754af09b6c82ba70472f181d03a7b3a919907
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling data

In [ ]:
# DATASET
from datasets import load_dataset

ds = load_dataset("abisee/cnn_dailymail", revision="refs/convert/parquet")
ds

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 861339
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 40104
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 34470
    })
})

In [ ]:
# TOKENIZER
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_ckpt = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [ ]:
# PREPROCESS
max_input = 512
max_target = 128

def preprocess(batch):
    inputs = batch["article"]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=batch["highlights"],
        max_length=max_target,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)
tokenized

Map:   0%|          | 0/861339 [00:00<?, ? examples/s]

Map:   0%|          | 0/40104 [00:00<?, ? examples/s]

Map:   0%|          | 0/34470 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 861339
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40104
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 34470
    })
})

In [ ]:
# ROUGE
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

rouge = evaluate.load("rouge")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/bart-cnn-full2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

checkpoint_path = str(OUTPUT_DIR / "checkpoint-50000")

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    do_train=True,
    do_eval=True,

    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    logging_steps=200,

    learning_rate=3e-5,
    warmup_steps=500,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    num_train_epochs=2,
    fp16=True,
    predict_with_generate=True,

    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"].shuffle(seed=42),
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=None,
)

trainer.train(resume_from_checkpoint=checkpoint_path)

final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving final model to {final_dir}")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss
52000,7.250545,1.709833
54000,7.041489,1.714487
56000,7.012556,1.710771
58000,6.834017,1.711910
60000,6.932899,1.712217
62000,6.945173,1.708227
64000,6.935620,1.706697
66000,6.859178,1.700448
68000,6.919647,1.706206
70000,6.912260,1.707884


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving final model to /content/drive/MyDrive/bart-cnn-full2/final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bart-cnn-full2/final/tokenizer_config.json',
 '/content/drive/MyDrive/bart-cnn-full2/final/tokenizer.json')

In [ ]:
import torch
from tqdm.auto import tqdm
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def eval_rouge(ds_split, n=500, num_beams=6, max_input=512, max_target=128):
    preds, refs = [], []
    n = min(n, len(ds_split))
    for i in tqdm(range(n)):
        ex = ds_split[i]
        inputs = tokenizer(
            ex["article"],
            return_tensors="pt",
            truncation=True,
            max_length=max_input
        ).to(model.device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=num_beams,
                max_length=max_target,
                min_length=30,
                length_penalty=2.0,
                early_stopping=True
            )

        preds.append(tokenizer.decode(summary_ids[0], skip_special_tokens=True))
        refs.append(ex["highlights"])

    return rouge.compute(predictions=preds, references=refs, use_stemmer=True)

val_scores_2k = eval_rouge(ds["validation"], n=2000, num_beams=4, max_input=max_input, max_target=max_target)
val_scores_2k

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1000,844.344844,108.389496,0.000000,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1000,844.344844,108.389496,0.000000,0.000000,0.000000,0.000000


In [ ]:
trainer.save_model("/kaggle/working/bart-cnn-subset")
tokenizer.save_pretrained("/kaggle/working/bart-cnn-subset")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/bart-cnn-subset/tokenizer_config.json',
 '/kaggle/working/bart-cnn-subset/tokenizer.json')

In [ ]:
import torch

i = 0

text = ds["test"][i]["article"]
reference = ds["test"][i]["highlights"]

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(model.device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        num_beams=4,
        max_length=128,
        min_length=30,
        length_penalty=2.0,
        early_stopping=True
    )

prediction = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("="*80)
print("ARTICLE (first 1000 chars):\n")
print(text[:1000] + "...\n")

print("="*80)
print("MODEL SUMMARY:\n")
print(prediction)

print("="*80)
print("REFERENCE SUMMARY:\n")
print(reference)

ARTICLE (first 1000 chars):

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speakin